# House Prices: BaseMLP baseline

## tl;dr

- 모델: 전처리된 입력 → 128 → 64 → 1, ReLU, dropout·batch normalization 없음
- 검증: `KFold(5, shuffle=True, random_state=42)` 및 fold별 best-checkpoint 복원
- 재현 결과: 5-fold RMSLE **0.150342 ± 0.040707**, OOF RMSLE **0.154688**
- 노트북 내부에 전처리·모델·학습·검증·추론 코드를 모두 포함한다.
- 입력은 `train.csv`, `test.csv`, `sample_submission.csv`만 사용한다.
- test predictor는 분석하지 않고 제출 형식 확인과 blind inference에만 사용한다.


## Context & Methods

이 노트북은 외부 프로젝트 Python 모듈에 의존하지 않는 독립형 BaseMLP 실험이다. 전처리는 각 fold의 학습 부분에서만 적합하고, validation RMSLE가 가장 낮은 checkpoint를 복원한 뒤 OOF와 test를 추론한다.

### Key Assumptions

- RMSLE는 `log1p(SalePrice)` 공간의 RMSE와 동일하게 계산한다.
- 노트북은 원본 입력 CSV 3개만 읽고 모델·전처리기·평가·submission을 직접 생성한다.
- test는 스키마·Id 순서 검증과 최종 blind inference 외에는 검사하거나 해석하지 않는다.
- 재실행은 동일한 독립 experiment ID의 notebook artifact를 갱신한다.


In [1]:
from __future__ import annotations

import hashlib
import json
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

root_candidates = (Path.cwd(), Path.cwd().parent)
ROOT = next(
    (candidate for candidate in root_candidates if (candidate / "data" / "train.csv").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from the project root or notebooks directory.")

TRAIN_PATH = ROOT / "data" / "train.csv"
TEST_PATH = ROOT / "data" / "test.csv"
SAMPLE_SUBMISSION_PATH = ROOT / "data" / "sample_submission.csv"
NOTEBOOK_EXPERIMENT_ID = "BASEMLP-20260720-STANDALONE-NB-01"
NOTEBOOK_SUBMISSION_PATH = ROOT / "submissions" / f"submission_{NOTEBOOK_EXPERIMENT_ID}.csv"
NOTEBOOK_OOF_PATH = ROOT / "reports" / f"basemlp_oof_{NOTEBOOK_EXPERIMENT_ID}.csv"
NOTEBOOK_FOLD_PREDICTIONS_PATH = ROOT / "reports" / f"basemlp_test_fold_predictions_{NOTEBOOK_EXPERIMENT_ID}.csv"
METRICS_PATH = ROOT / "reports" / f"basemlp_metrics_{NOTEBOOK_EXPERIMENT_ID}.json"
SUBMISSION_VALIDATION_PATH = ROOT / "reports" / f"basemlp_submission_validation_{NOTEBOOK_EXPERIMENT_ID}.json"

SEED = 42
N_SPLITS = 5
HIDDEN_DIMS = (128, 64)
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0
BATCH_SIZE = 64
MAX_EPOCHS = 200
PATIENCE = 25
MIN_DELTA = 1e-5
DEVICE = torch.device("cpu")

NUMERIC_COLUMNS = [
    "LotFrontage", "LotArea", "OverallQual", "OverallCond", "YearBuilt",
    "YearRemodAdd", "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF",
    "TotalBsmtSF", "1stFlrSF", "2ndFlrSF", "LowQualFinSF", "GrLivArea",
    "BsmtFullBath", "BsmtHalfBath", "FullBath", "HalfBath", "BedroomAbvGr",
    "KitchenAbvGr", "TotRmsAbvGrd", "Fireplaces", "GarageYrBlt", "GarageCars",
    "GarageArea", "WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch",
    "ScreenPorch", "PoolArea", "MiscVal", "YrSold",
]
BASEMENT_COLUMNS = ["BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]
GARAGE_COLUMNS = ["GarageType", "GarageFinish", "GarageQual", "GarageCond"]

train = pd.read_csv(TRAIN_PATH, keep_default_na=False)
test = pd.read_csv(TEST_PATH, keep_default_na=False)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
categorical_columns = [
    column
    for column in train.columns
    if column not in {*NUMERIC_COLUMNS, "Id", "SalePrice"}
]

assert train.shape == (1460, 81)
assert test.shape == (1459, 80)
assert list(test.columns) == [column for column in train.columns if column != "SalePrice"]
assert list(sample_submission.columns) == ["Id", "SalePrice"]
assert sample_submission["Id"].equals(test["Id"])
assert len(NUMERIC_COLUMNS) == 34 and len(categorical_columns) == 45
print(f"train={train.shape}, test={test.shape}, categorical={len(categorical_columns)}")


train=(1460, 81), test=(1459, 80), categorical=45


## Model & Preprocessing

수치형은 fold median imputation과 표준화, 범주형은 구조적 부재 라벨과 fold one-hot encoding을 사용한다. 타깃은 fold 학습 부분의 평균·표준편차로 표준화한 `log1p(SalePrice)`다.

In [2]:
class BaseMLP(nn.Module):
    """Small two-hidden-layer tabular MLP."""

    def __init__(self, input_dim: int) -> None:
        super().__init__()
        self.hidden1 = nn.Linear(input_dim, HIDDEN_DIMS[0])
        self.hidden2 = nn.Linear(HIDDEN_DIMS[0], HIDDEN_DIMS[1])
        self.output = nn.Linear(HIDDEN_DIMS[1], 1)
        self.activation = nn.ReLU()
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.kaiming_normal_(self.hidden1.weight, nonlinearity="relu")
        nn.init.zeros_(self.hidden1.bias)
        nn.init.kaiming_normal_(self.hidden2.weight, nonlinearity="relu")
        nn.init.zeros_(self.hidden2.bias)
        nn.init.xavier_normal_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        hidden = self.activation(self.hidden1(inputs))
        hidden = self.activation(self.hidden2(hidden))
        return self.output(hidden).squeeze(1)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def clean_rows(frame: pd.DataFrame, categorical_columns: list[str]) -> pd.DataFrame:
    cleaned = frame.copy()

    for column in NUMERIC_COLUMNS:
        cleaned[column] = pd.to_numeric(
            cleaned[column].replace({"NA": np.nan, "": np.nan}),
            errors="coerce",
        )

    # Preserve comparability with the fixed linear/MLP baseline setup.
    cleaned.loc[cleaned["Id"].eq(524), "YearRemodAdd"] = 2007

    basement_absent = cleaned["TotalBsmtSF"].fillna(0).eq(0)
    garage_absent = (
        cleaned["GarageCars"].fillna(0).eq(0)
        & cleaned["GarageArea"].fillna(0).eq(0)
    )

    for column in BASEMENT_COLUMNS:
        missing = cleaned[column].isin(["NA", ""])
        cleaned.loc[missing & basement_absent, column] = "NoBasement"
        cleaned.loc[missing & ~basement_absent, column] = "Unknown"

    for column in GARAGE_COLUMNS:
        missing = cleaned[column].isin(["NA", ""])
        cleaned.loc[missing & garage_absent, column] = "NoGarage"
        cleaned.loc[missing & ~garage_absent, column] = "Unknown"
    cleaned.loc[garage_absent, "GarageYrBlt"] = 0.0

    fireplace_absent = cleaned["Fireplaces"].fillna(0).eq(0)
    pool_absent = cleaned["PoolArea"].fillna(0).eq(0)
    cleaned.loc[
        cleaned["FireplaceQu"].isin(["NA", ""]) & fireplace_absent,
        "FireplaceQu",
    ] = "NoFireplace"
    cleaned.loc[
        cleaned["PoolQC"].isin(["NA", ""]) & pool_absent,
        "PoolQC",
    ] = "NoPool"

    for column, label in {
        "Alley": "NoAlley",
        "Fence": "NoFence",
        "MiscFeature": "NoMiscFeature",
    }.items():
        cleaned[column] = cleaned[column].replace({"NA": label, "": label})

    cleaned["MSSubClass"] = cleaned["MSSubClass"].map(lambda value: f"SC{value}")
    cleaned["MoSold"] = cleaned["MoSold"].map(lambda value: f"M{int(value):02d}")

    for column in categorical_columns:
        cleaned[column] = cleaned[column].replace({"NA": "Unknown", "": "Unknown"})
    return cleaned


def make_preprocessor(categorical_columns: list[str]) -> ColumnTransformer:
    numeric = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )
    return ColumnTransformer(
        [
            ("numeric", numeric, NUMERIC_COLUMNS),
            ("categorical", categorical, categorical_columns),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )


def predict_log_target(
    model: BaseMLP,
    features: np.ndarray,
    target_mean: float,
    target_std: float,
) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        tensor = torch.as_tensor(features, dtype=torch.float32, device=DEVICE)
        standardized = model(tensor).cpu().numpy().astype(np.float64)
    return standardized * target_std + target_mean


def train_fold(
    X_train: np.ndarray,
    y_train_log: np.ndarray,
    X_valid: np.ndarray,
    y_valid_log: np.ndarray,
    seed: int,
    experiment_id: str,
    fold: int,
    checkpoint_path: Path,
    epoch_log_path: Path,
) -> tuple[BaseMLP, dict[str, float | int]]:
    set_seed(seed)
    target_mean = float(y_train_log.mean())
    target_std = float(y_train_log.std(ddof=0))
    y_train_standardized = (y_train_log - target_mean) / target_std
    y_valid_standardized = (y_valid_log - target_mean) / target_std

    train_dataset = TensorDataset(
        torch.as_tensor(X_train, dtype=torch.float32),
        torch.as_tensor(y_train_standardized, dtype=torch.float32),
    )
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
    )
    X_valid_tensor = torch.as_tensor(X_valid, dtype=torch.float32, device=DEVICE)
    y_valid_tensor = torch.as_tensor(y_valid_standardized, dtype=torch.float32, device=DEVICE)

    model = BaseMLP(X_train.shape[1]).to(DEVICE)
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    best_score = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0
    epoch_rows: list[dict[str, float | int]] = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0
        train_count = 0

        for batch_features, batch_target in train_loader:
            batch_features = batch_features.to(DEVICE)
            batch_target = batch_target.to(DEVICE)
            optimizer.zero_grad()
            prediction = model(batch_features)
            loss = loss_fn(prediction, batch_target)
            loss.backward()
            optimizer.step()
            train_loss_sum += float(loss.detach().cpu()) * len(batch_features)
            train_count += len(batch_features)

        model.eval()
        with torch.no_grad():
            valid_standardized = model(X_valid_tensor)
            valid_loss = loss_fn(valid_standardized, y_valid_tensor)
            valid_log_prediction = (
                valid_standardized.cpu().numpy().astype(np.float64) * target_std
                + target_mean
            )

        valid_rmsle = float(np.sqrt(np.mean((y_valid_log - valid_log_prediction) ** 2)))
        epoch_rows.append(
            {
                "epoch": epoch,
                "train_loss": train_loss_sum / train_count,
                "validation_loss": float(valid_loss.cpu()),
                "validation_rmsle": valid_rmsle,
                "learning_rate": float(optimizer.param_groups[0]["lr"]),
            }
        )

        if valid_rmsle < best_score - MIN_DELTA:
            best_score = valid_rmsle
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save(
                {
                    "experiment_id": experiment_id,
                    "fold": fold,
                    "input_dim": int(X_train.shape[1]),
                    "model_state_dict": model.state_dict(),
                    "target_mean": target_mean,
                    "target_std": target_std,
                    "architecture": [*HIDDEN_DIMS, 1],
                    "seed": seed,
                },
                checkpoint_path,
            )
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            break

    pd.DataFrame(epoch_rows).to_csv(epoch_log_path, index=False)

    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    restored_prediction = predict_log_target(
        model,
        X_valid,
        checkpoint["target_mean"],
        checkpoint["target_std"],
    )
    restored_rmsle = float(np.sqrt(np.mean((y_valid_log - restored_prediction) ** 2)))
    if not np.isclose(restored_rmsle, best_score, rtol=0, atol=1e-6):
        raise RuntimeError("Restored checkpoint score does not match the saved best score.")

    return model, {
        "best_epoch": best_epoch,
        "stopping_epoch": epoch,
        "best_validation_rmsle": best_score,
        "restored_validation_rmsle": restored_rmsle,
        "target_mean": target_mean,
        "target_std": target_std,
    }


display(pd.Series({
    "hidden_dims": HIDDEN_DIMS,
    "activation": "ReLU",
    "dropout": 0.0,
    "batch_normalization": False,
    "optimizer": "Adam",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "loss": "MSELoss",
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "device": str(DEVICE),
}, name="BaseMLP configuration"))


hidden_dims            (128, 64)
activation                  ReLU
dropout                      0.0
batch_normalization        False
optimizer                   Adam
learning_rate              0.001
weight_decay                 0.0
loss                     MSELoss
batch_size                    64
max_epochs                   200
patience                      25
device                       cpu
Name: BaseMLP configuration, dtype: object

## Data

학습 입력과 target을 준비한다. test는 같은 row-only cleaning을 적용한 뒤 fold별 blind inference에만 전달한다.

In [3]:
X = clean_rows(train.drop(columns="SalePrice"), categorical_columns)
X_test = clean_rows(test, categorical_columns)
y = train["SalePrice"].to_numpy(dtype=np.float64)
y_log = np.log1p(y)

checkpoint_dir = ROOT / "artifacts" / "checkpoints" / NOTEBOOK_EXPERIMENT_ID
preprocessor_dir = ROOT / "artifacts" / "preprocessors" / NOTEBOOK_EXPERIMENT_ID
epoch_log_dir = ROOT / "artifacts" / "logs" / NOTEBOOK_EXPERIMENT_ID
for directory in (checkpoint_dir, preprocessor_dir, epoch_log_dir):
    directory.mkdir(parents=True, exist_ok=True)

print(f"model inputs={X.shape}, target={y_log.shape}")


model inputs=(1460, 80), target=(1460,)


## Results

각 fold에서 전처리기와 모델을 새로 적합하고, 저장한 best checkpoint와 전처리기를 다시 불러와 OOF와 test를 추론한다.

In [4]:
kfold = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_log_prediction = np.full(len(X), np.nan, dtype=np.float64)
test_log_predictions = []
fold_rows = []

for fold, (train_index, valid_index) in enumerate(kfold.split(X), start=1):
    fold_seed = SEED + fold
    checkpoint_path = checkpoint_dir / f"fold_{fold}_best.pt"
    preprocessor_path = preprocessor_dir / f"fold_{fold}_preprocessor.joblib"
    epoch_log_path = epoch_log_dir / f"fold_{fold}_epochs.csv"

    preprocessor = make_preprocessor(categorical_columns)
    X_train = preprocessor.fit_transform(X.iloc[train_index]).astype(np.float32)
    X_valid = preprocessor.transform(X.iloc[valid_index]).astype(np.float32)
    joblib.dump(preprocessor, preprocessor_path)

    _, training = train_fold(
        X_train,
        y_log[train_index],
        X_valid,
        y_log[valid_index],
        seed=fold_seed,
        experiment_id=NOTEBOOK_EXPERIMENT_ID,
        fold=fold,
        checkpoint_path=checkpoint_path,
        epoch_log_path=epoch_log_path,
    )

    saved_preprocessor = joblib.load(preprocessor_path)
    X_valid_saved = saved_preprocessor.transform(X.iloc[valid_index]).astype(np.float32)
    X_test_saved = saved_preprocessor.transform(X_test).astype(np.float32)
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    restored_model = BaseMLP(checkpoint["input_dim"]).to(DEVICE)
    restored_model.load_state_dict(checkpoint["model_state_dict"])

    valid_prediction = predict_log_target(
        restored_model, X_valid_saved, checkpoint["target_mean"], checkpoint["target_std"]
    )
    test_prediction = predict_log_target(
        restored_model, X_test_saved, checkpoint["target_mean"], checkpoint["target_std"]
    )
    oof_log_prediction[valid_index] = valid_prediction
    test_log_predictions.append(test_prediction)

    fold_rmsle = float(np.sqrt(np.mean((y_log[valid_index] - valid_prediction) ** 2)))
    fold_rows.append({
        "fold": fold,
        "seed": fold_seed,
        "encoded_features": X_train.shape[1],
        "best_epoch": training["best_epoch"],
        "stopping_epoch": training["stopping_epoch"],
        "rmsle": fold_rmsle,
    })
    print(
        f"fold {fold}: RMSLE={fold_rmsle:.6f}, "
        f"best={training['best_epoch']}, stop={training['stopping_epoch']}"
    )

assert not np.isnan(oof_log_prediction).any()
fold_results = pd.DataFrame(fold_rows)
display(fold_results)


fold 1: RMSLE=0.140891, best=12, stop=37


fold 2: RMSLE=0.131788, best=43, stop=68


fold 3: RMSLE=0.222199, best=5, stop=30


fold 4: RMSLE=0.134316, best=21, stop=46


fold 5: RMSLE=0.122514, best=14, stop=39


,fold,seed,encoded_features,best_epoch,stopping_epoch,rmsle
0,1,43,327,12,37,0.140891
1,2,44,327,43,68,0.131788
2,3,45,323,5,30,0.222199
3,4,46,324,21,46,0.134316
4,5,47,328,14,39,0.122514


In [5]:
fold_scores = fold_results["rmsle"].to_numpy(dtype=np.float64)
cv_mean = float(fold_scores.mean())
cv_std = float(fold_scores.std(ddof=1))
oof_price_prediction = np.clip(np.expm1(oof_log_prediction), 0, None)
oof_rmsle = float(np.sqrt(np.mean((y_log - np.log1p(oof_price_prediction)) ** 2)))

test_log_prediction_matrix = np.vstack(test_log_predictions)
mean_test_log_prediction = test_log_prediction_matrix.mean(axis=0)
test_price_prediction = np.clip(np.expm1(mean_test_log_prediction), 0, None)

pd.DataFrame({
    "Id": train["Id"],
    "SalePrice": y,
    "oof_log_prediction": oof_log_prediction,
    "oof_saleprice_prediction": oof_price_prediction,
}).to_csv(NOTEBOOK_OOF_PATH, index=False)

pd.DataFrame({
    "Id": test["Id"],
    **{
        f"fold_{fold}_log_prediction": test_log_prediction_matrix[fold - 1]
        for fold in range(1, N_SPLITS + 1)
    },
    "mean_log_prediction": mean_test_log_prediction,
    "SalePrice": test_price_prediction,
}).to_csv(NOTEBOOK_FOLD_PREDICTIONS_PATH, index=False)

notebook_submission = sample_submission.copy()
notebook_submission["SalePrice"] = test_price_prediction
NOTEBOOK_SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
notebook_submission.to_csv(NOTEBOOK_SUBMISSION_PATH, index=False)

display(pd.Series({
    "cv_mean": cv_mean,
    "cv_std": cv_std,
    "oof_rmsle": oof_rmsle,
    "submission_rows": len(notebook_submission),
}, name="Standalone notebook results"))


cv_mean               0.150342
cv_std                0.040707
oof_rmsle             0.154688
submission_rows    1459.000000
Name: Standalone notebook results, dtype: float64

## Submission Validation

방금 노트북이 직접 생성한 submission을 다시 읽어 열·dtype·Id 순서와 예측 유효성을 검증한다. 개별 test 예측은 표시하거나 해석하지 않는다.


In [6]:
def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


reloaded_submission = pd.read_csv(NOTEBOOK_SUBMISSION_PATH)
checks = {
    "columns_equal": reloaded_submission.columns.tolist() == ["Id", "SalePrice"],
    "rows_equal": len(reloaded_submission) == len(test),
    "dtypes_equal": (
        reloaded_submission.dtypes.astype(str).to_dict()
        == sample_submission.dtypes.astype(str).to_dict()
    ),
    "id_exact_equal": reloaded_submission["Id"].equals(test["Id"]),
    "unique_ids": reloaded_submission["Id"].is_unique,
    "missing_predictions": int(reloaded_submission["SalePrice"].isna().sum()) == 0,
    "finite_predictions": bool(np.isfinite(reloaded_submission["SalePrice"]).all()),
    "positive_predictions": bool(reloaded_submission["SalePrice"].gt(0).all()),
    "fold_artifacts": all(
        len(list(directory.glob(pattern))) == N_SPLITS
        for directory, pattern in (
            (checkpoint_dir, "fold_*_best.pt"),
            (preprocessor_dir, "fold_*_preprocessor.joblib"),
            (epoch_log_dir, "fold_*_epochs.csv"),
        )
    ),
}

submission_validation = {
    "experiment_id": NOTEBOOK_EXPERIMENT_ID,
    "submission_path": str(NOTEBOOK_SUBMISSION_PATH.relative_to(ROOT)),
    "submission_sha256": sha256(NOTEBOOK_SUBMISSION_PATH),
    "checks": checks,
    "all_checks_passed": all(checks.values()),
}
SUBMISSION_VALIDATION_PATH.write_text(
    json.dumps(submission_validation, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

metrics = {
    "experiment_id": NOTEBOOK_EXPERIMENT_ID,
    "source": {
        "train_path": str(TRAIN_PATH.relative_to(ROOT)),
        "train_sha256": sha256(TRAIN_PATH),
        "test_path": str(TEST_PATH.relative_to(ROOT)),
        "test_sha256": sha256(TEST_PATH),
        "sample_submission_path": str(SAMPLE_SUBMISSION_PATH.relative_to(ROOT)),
        "sample_submission_sha256": sha256(SAMPLE_SUBMISSION_PATH),
    },
    "preprocessing": {
        "numeric": "fold median imputation and StandardScaler",
        "categorical": "row-only structural absence labels; fold one-hot handle_unknown=ignore",
        "target": "fold-train mean/std standardization of log1p(SalePrice)",
    },
    "validation": {
        "splitter": "KFold",
        "n_splits": N_SPLITS,
        "shuffle": True,
        "random_state": SEED,
        "metric": "RMSLE / RMSE on log1p target",
    },
    "model": {
        "class": "BaseMLP(nn.Module)",
        "architecture": [*HIDDEN_DIMS, 1],
        "activation": "ReLU",
        "optimizer": "Adam",
        "learning_rate": LEARNING_RATE,
        "loss": "MSELoss on fold-standardized log1p(SalePrice)",
        "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "seed": SEED,
        "device": str(DEVICE),
    },
    "results": {
        "fold_scores": fold_scores.tolist(),
        "cv_mean": cv_mean,
        "cv_std": cv_std,
        "oof_rmsle": oof_rmsle,
        "best_epochs": fold_results["best_epoch"].tolist(),
        "stopping_epochs": fold_results["stopping_epoch"].tolist(),
        "test_prediction_strategy": "mean of 5 restored-checkpoint log predictions, then expm1",
    },
    "artifacts": {
        "notebook": "notebooks/house_prices_basemlp.ipynb",
        "submission": str(NOTEBOOK_SUBMISSION_PATH.relative_to(ROOT)),
        "oof_predictions": str(NOTEBOOK_OOF_PATH.relative_to(ROOT)),
        "test_fold_predictions": str(NOTEBOOK_FOLD_PREDICTIONS_PATH.relative_to(ROOT)),
        "submission_validation": str(SUBMISSION_VALIDATION_PATH.relative_to(ROOT)),
    },
}
METRICS_PATH.write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

display(pd.Series(checks, name="Submission checks"))
assert submission_validation["all_checks_passed"], submission_validation


columns_equal           True
rows_equal              True
dtypes_equal            True
id_exact_equal          True
unique_ids              True
missing_predictions     True
finite_predictions      True
positive_predictions    True
fold_artifacts          True
Name: Submission checks, dtype: bool

## Takeaways

- 6개 code cell이 외부 프로젝트 Python 모듈 없이 top-to-bottom 실행된다.
- 입력은 원본 CSV 3개뿐이며 전처리·BaseMLP·학습 루프·checkpoint 복원·추론·submission 생성을 노트북 안에서 수행한다.
- 실행 결과와 설정은 `reports/basemlp_metrics_BASEMLP-20260720-STANDALONE-NB-01.json`에 저장한다.
- 생성 submission은 검증용 독립 산출물이며 Kaggle에 새로 제출한 파일은 아니다.
